# Time integration

In [ ]:
from lucifex.fdm import CN
from lucifex.fem import Constant
from lucifex.sim import (
    run, stopper, writer, 
)
from lucifex.plt import plot_line
from py.diffusion import diffusion_simulation_1d


simulation = diffusion_simulation_1d(store_delta=1)(
    Lx=1.0,
    Nx=200,
    dt=0.123,
    m_exponent=2,
    D_fdm=CN,
)

n_stop = 30
run(simulation, n_stop=n_stop)
u = simulation['u']

print(f'n_stop = {n_stop}')
print(f'length of stored series = {len(u.series)}')

## Stopping criteria

In [ ]:
def is_less_than(
    u: Constant,
    u_thresh: float,
) -> bool:
    return u.value < u_thresh

simulation = diffusion_simulation_1d(store_delta=1)(
    Lx=1.0,
    Nx=200,
    dt=0.001,
    m_exponent=2,
    D_fdm=CN,
)
uMax = simulation['uMax']

uMax_thresh = 0.5
uMaX_stopper = stopper(uMax[-1], is_less_than)(uMax_thresh)

n_stop = 500
run(simulation, n_stop=n_stop, stoppers=[stopper])

plot_line((uMax.time_series, uMax.value_series))

## Writing criteria

In [ ]:
def is_in_range(
    u: Constant,
    u_low: float,
    u_high: float,
) -> bool:
    return u.value < u_high and u.value > u_low

simulation = diffusion_simulation_1d(store_delta=1)(
    Lx=1.0,
    Nx=200,
    dt=0.001,
    m_exponent=2,
    D_fdm=CN,
)
name = 'uMax'
uMax = simulation['uMax']

u_high = 0.8
u_low = 0.6
routine = lambda t, u: print(f'Is in range: {name}={float(u)} when t={t}')
uMax_writer_func = writer(uMax[-1], is_in_range, routine)(u_low, u_high)

n_step = 10
routine = lambda t, u: print(f'Every {n_step} steps: {name}={float(u)} when t={t}')
uMax_writer_int = writer(uMax[0], n_step, routine)

dt_step = 0.025
routine = lambda t, u: print(f'Every {dt_step} time units: {name}={float(u)} when t={t}')
uMax_writer_float = writer(uMax[0], dt_step, routine)

n_stop = 100
run(simulation, n_stop=n_stop, writers=[uMax_writer_func, uMax_writer_int, uMax_writer_float])
